# Llama3 KerasHub → HuggingFace Export Verification

This notebook verifies that Llama3 models exported from KerasHub to HuggingFace format match the original HuggingFace models.

## What this notebook does:
1. ✅ Loads a KerasHub Llama3 preset
2. ✅ Exports it to HuggingFace format
3. ✅ Compares configs, parameters, logits, and generation with the original HF model
4. ✅ Reports validation results

## Supported Models:
- **Llama 3.0**: `llama3_8b_en`, `llama3_instruct_8b_en`
- **Llama 3.1**: `llama3.1_8b`, `llama3.1_instruct_8b`, `llama3.1_guard_8b`
- **Llama 3.2**: `llama3.2_1b`, `llama3.2_instruct_1b`, `llama3.2_3b`, `llama3.2_instruct_3b`, `llama3.2_guard_1b`

> **Note:** All Llama3 models on HuggingFace are gated. You must [accept the license on HuggingFace](https://huggingface.co/meta-llama) before downloading them.

## Step 1: Install Dependencies

Install required packages (takes ~2-3 minutes).

In [ ]:
!pip install -q keras-hub transformers torch safetensors huggingface_hub
print("✅ Dependencies installed!")

## Step 2: Set Up HuggingFace Credentials

Llama3 models are gated and require a HuggingFace token.

**Instructions:**
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a **Read** token
3. Accept the Llama3 license at [huggingface.co/meta-llama](https://huggingface.co/meta-llama)
4. Either:
   - Add `HF_TOKEN` to Colab Secrets (recommended), **or**
   - Paste your token directly in the cell below

In [ ]:
import os

from huggingface_hub import login

# Option A: Use Colab Secrets (recommended)
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Logged in via Colab Secrets!")
except Exception:
    # Option B: Paste token directly (less secure)
    # hf_token = "hf_YOUR_TOKEN_HERE"
    # login(token=hf_token)
    print("⚠ Could not read HF_TOKEN from Colab Secrets.")
    print("  Uncomment Option B above and paste your token, or run: login()")

## Step 3: Set Keras Backend

Configure Keras to use PyTorch backend.

In [ ]:
os.environ["KERAS_BACKEND"] = "torch"

import keras

print(f"✅ Keras backend: {keras.config.backend()}")
print(f"✅ Keras version: {keras.__version__}")

## Step 4: Load Verification Functions

Run this cell to load all verification logic.

In [ ]:
import gc
import os
import tempfile
from numbers import Number

import numpy as np
import torch
from transformers import AutoConfig
from transformers import AutoTokenizer
from transformers import LlamaForCausalLM

from keras_hub.src.models.llama3.llama3_causal_lm import Llama3CausalLM

# -------------------------------------------------------------------
# Constants
# -------------------------------------------------------------------

PRESET_TO_HF = {
    # Llama 3.0
    "llama3_8b_en": "meta-llama/Meta-Llama-3-8B",
    "llama3_instruct_8b_en": "meta-llama/Meta-Llama-3-8B-Instruct",
    # Llama 3.1
    "llama3.1_8b": "meta-llama/Llama-3.1-8B",
    "llama3.1_instruct_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama3.1_guard_8b": "meta-llama/Llama-Guard-3.1-8B",
    # Llama 3.2
    "llama3.2_1b": "meta-llama/Llama-3.2-1B",
    "llama3.2_instruct_1b": "meta-llama/Llama-3.2-1B-Instruct",
    "llama3.2_3b": "meta-llama/Llama-3.2-3B",
    "llama3.2_instruct_3b": "meta-llama/Llama-3.2-3B-Instruct",
    "llama3.2_guard_1b": "meta-llama/Llama-Guard-3.2-1B",
}

TEXT_PROMPT = "The capital of France is"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# -------------------------------------------------------------------
# Config comparison
# -------------------------------------------------------------------


def configs_equal(val1, val2, tolerance=1e-6):
    """Compare two config values with tolerance for numeric differences."""
    if isinstance(val1, bool) or isinstance(val2, bool):
        return val1 == val2
    if isinstance(val1, Number) and isinstance(val2, Number):
        return abs(float(val1) - float(val2)) < tolerance
    if not isinstance(val1, type(val2)):
        return False
    if isinstance(val1, (str, type(None))):
        return val1 == val2
    if isinstance(val1, (list, tuple)):
        if len(val1) != len(val2):
            return False
        return all(
            configs_equal(v1, v2, tolerance) for v1, v2 in zip(val1, val2)
        )
    if isinstance(val1, dict):
        if set(val1.keys()) != set(val2.keys()):
            return False
        return all(configs_equal(val1[k], val2.get(k), tolerance) for k in val1)
    return val1 == val2


def classify_difference(key, orig_val, exp_val):
    """Return (is_non_critical, reason) for a config key mismatch."""
    orig_missing = orig_val == "<missing>"
    exp_missing = exp_val == "<missing>"

    # rope_scaling float sub-fields may differ by epsilon across versions
    if (
        key == "rope_scaling"
        and isinstance(orig_val, dict)
        and isinstance(exp_val, dict)
    ):
        if configs_equal(orig_val, exp_val, tolerance=1e-4):
            return True, "rope_scaling within float tolerance"

    # Fields only present in newer transformers versions
    non_critical_keys = {
        "rope_scaling",  # may have extra keys in newer HF versions
    }
    if key in non_critical_keys and (orig_missing or exp_missing):
        return True, "optional/version-dependent field"

    # KerasHub always stores embedding matrices as untied (False) even
    # when the original model had tie_word_embeddings=True. The exported
    # weights are numerically identical; only the storage convention differs.
    if key == "tie_word_embeddings" and orig_val is True and exp_val is False:
        return (
            True,
            "KerasHub stores untied embeddings by convention"
            " (logits are identical)",
        )

    return False, ""


# -------------------------------------------------------------------
# Phase 1: Export
# -------------------------------------------------------------------


def export_keras_model(preset, export_path):
    """Load KerasHub preset and export to HF format."""
    print("\n[1/6] Loading KerasHub model from preset...")
    try:
        keras_model = Llama3CausalLM.from_preset(preset)
    except Exception as e:
        print(f"  ✗ Failed to load KerasHub model: {e}")
        raise

    backbone = keras_model.backbone
    print(
        f"  ✓ Loaded: {backbone.num_layers} layers, "
        f"{backbone.hidden_dim}d, {backbone.vocabulary_size} vocab"
    )
    print(f"  ✓ Parameters: {keras_model.count_params():,}")

    print(f"\n[2/6] Exporting to HF format → {export_path}...")
    try:
        keras_model.export_to_transformers(export_path)
    except Exception as e:
        print(f"  ✗ Export failed: {e}")
        raise

    all_exist = True
    for fname in [
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
    ]:
        fpath = os.path.join(export_path, fname)
        exists = os.path.exists(fpath)
        size = os.path.getsize(fpath) if exists else 0
        print(f"  {'✓' if exists else '✗'} {fname} ({size:,} bytes)")
        if not exists:
            all_exist = False

    if not all_exist:
        raise FileNotFoundError("Some expected export files are missing")

    del keras_model
    gc.collect()


# -------------------------------------------------------------------
# Phase 2: Precompute original outputs
# -------------------------------------------------------------------


def precompute_original_outputs(hf_model_id, skip_generation):
    """Load original HF model and precompute outputs."""
    print(f"\n[3/6] Loading ORIGINAL HF model: {hf_model_id}...")
    try:
        hf_model = LlamaForCausalLM.from_pretrained(
            hf_model_id, torch_dtype=torch.float32
        )
        hf_tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
    except Exception as e:
        print(f"  ✗ Failed to load HF model: {e}")
        raise

    hf_model.eval().to(device)
    hf_params = sum(p.numel() for p in hf_model.parameters())
    print(f"  ✓ Original HF: {hf_params:,} parameters")

    results = {"hf_params": hf_params}

    print("\n  Computing text logits...")
    hf_inputs = hf_tokenizer(TEXT_PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        text_out = hf_model(**hf_inputs)
    results["text_logits"] = text_out.logits.float().cpu().numpy()
    results["text_input_ids"] = hf_inputs["input_ids"].cpu().numpy()
    print(f"    Text logits shape: {results['text_logits'].shape}")

    if not skip_generation:
        with torch.no_grad():
            gen_out = hf_model.generate(
                **hf_inputs, max_new_tokens=30, do_sample=False
            )
        results["text_generated"] = hf_tokenizer.decode(
            gen_out[0][hf_inputs["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )
        print(f'    Text generation: "{results["text_generated"][:80]}"')

    del hf_model, hf_tokenizer
    gc.collect()
    return results


# -------------------------------------------------------------------
# Phase 3: Validate
# -------------------------------------------------------------------


def validate_configs(exp_cfg, orig_cfg):
    """Compare config fields and classify critical vs non-critical deltas."""
    print("\n  CONFIG VALIDATION")

    orig_dict = orig_cfg.to_dict() if hasattr(orig_cfg, "to_dict") else {}
    exp_dict = exp_cfg.to_dict() if hasattr(exp_cfg, "to_dict") else {}

    skip_keys = {
        "_name_or_path",
        "_attn_implementation",
        "_attn_implementation_autoset",
        "_commit_hash",
        "transformers_version",
        "torch_dtype",
        "auto_map",
        "architectures",
        "dtype",
    }

    all_keys = sorted(set(orig_dict.keys()) | set(exp_dict.keys()))
    config_pass = True
    critical_diffs, warning_diffs = [], []

    for key in all_keys:
        if key in skip_keys:
            continue
        o = orig_dict.get(key, "<missing>")
        e = exp_dict.get(key, "<missing>")
        if configs_equal(o, e):
            print(f"    ✓ {key}: {o}")
            continue
        is_non_critical, reason = classify_difference(key, o, e)
        if is_non_critical:
            print(f"    ⚠ {key}: original={o}, exported={e} ({reason})")
            warning_diffs.append(key)
        else:
            print(f"    ✗ {key}: original={o}, exported={e}")
            critical_diffs.append(key)
            config_pass = False

    if warning_diffs:
        print(
            f"\n    ⚠ {len(warning_diffs)} non-critical"
            f" field difference(s): {warning_diffs}"
        )
    if critical_diffs:
        print(
            f"\n    ✗ {len(critical_diffs)} critical"
            f" field difference(s): {critical_diffs}"
        )
    elif not warning_diffs:
        print(
            f"\n    ✓ All {len(all_keys) - len(skip_keys)} config fields match"
        )

    return config_pass


def validate_token_ids(exp_cfg, orig_cfg):
    """Compare special token IDs between exported and original models."""
    print("\n  TOKEN ID VALIDATION")

    token_fields = ["bos_token_id", "eos_token_id", "pad_token_id"]
    for attr in dir(orig_cfg):
        if attr.endswith("_token_id") and attr not in token_fields:
            token_fields.append(attr)

    token_pass = True
    for name in sorted(set(token_fields)):
        o = getattr(orig_cfg, name, None)
        e = getattr(exp_cfg, name, None)
        match = o == e
        print(f"    {'✓' if match else '✗'} {name}: original={o}, exported={e}")
        if not match:
            token_pass = False

    return token_pass


def validate_numerics(
    exp_model, exp_tokenizer, original_results, skip_generation
):
    """Compare logits and generation between exported and original models."""
    results = {}

    print("\n  TEXT LOGIT VALIDATION")
    text_ids = torch.tensor(original_results["text_input_ids"]).to(device)
    with torch.no_grad():
        exp_text_out = exp_model(input_ids=text_ids)
    exp_text_logits = exp_text_out.logits.float().cpu().numpy()
    orig_text_logits = original_results["text_logits"]

    text_diff = np.abs(exp_text_logits - orig_text_logits)
    results["text_mean_diff"] = float(text_diff.mean())
    print(f"    Logit mean abs diff: {results['text_mean_diff']:.2e}")
    results["text_pass"] = results["text_mean_diff"] < 0.1

    orig_top5 = set(np.argsort(orig_text_logits[0, -1])[-5:].tolist())
    exp_top5 = set(np.argsort(exp_text_logits[0, -1])[-5:].tolist())
    overlap = len(orig_top5 & exp_top5)
    print(f"    Top-5 token overlap: {overlap}/5 ({100 * overlap / 5:.0f}%)")

    if not skip_generation:
        print("\n  GENERATION COMPARISON")
        hf_inputs = exp_tokenizer(TEXT_PROMPT, return_tensors="pt").to(device)
        prompt_len = hf_inputs["input_ids"].shape[1]
        with torch.no_grad():
            exp_gen = exp_model.generate(
                **hf_inputs, max_new_tokens=30, do_sample=False
            )
        exp_gen_text = exp_tokenizer.decode(
            exp_gen[0][prompt_len:], skip_special_tokens=True
        )
        orig_gen_text = original_results.get("text_generated", "N/A")

        print(f'    Prompt:   "{TEXT_PROMPT}"')
        print(f'    Original: "{orig_gen_text[:80]}"')
        print(f'    Exported: "{exp_gen_text[:80]}"')

        if orig_gen_text == exp_gen_text:
            results["text_gen_match"] = True
            print("    ✓ Generation is IDENTICAL")
        else:
            results["text_gen_match"] = False
            print("    ⚠ Generation differs (expected with bf16→f32)")

    return results


def validate_exported_model(
    export_path, hf_model_id, original_results, skip_generation
):
    """Load the exported model and compare against original."""
    print(f"\n[4/6] Loading EXPORTED model from {export_path}...")
    try:
        exp_model = LlamaForCausalLM.from_pretrained(
            export_path, torch_dtype=torch.float32
        )
        exp_tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
    except Exception as e:
        print(f"  ✗ Failed to load exported model: {e}")
        raise

    exp_model.eval().to(device)
    exp_params = sum(p.numel() for p in exp_model.parameters())
    orig_params = original_results["hf_params"]
    print(f"  ✓ Exported: {exp_params:,} parameters")

    param_match = orig_params == exp_params
    print(
        f"  {'✓' if param_match else '✗'} Param count: "
        f"original={orig_params:,}, exported={exp_params:,}"
    )

    print("\n[5/6] Validating configs and token IDs...")
    orig_cfg = AutoConfig.from_pretrained(hf_model_id)
    exp_cfg = exp_model.config

    config_pass = validate_configs(exp_cfg, orig_cfg)
    token_pass = validate_token_ids(exp_cfg, orig_cfg)

    print("\n[6/6] Validating numerics...")
    numeric_results = validate_numerics(
        exp_model, exp_tokenizer, original_results, skip_generation
    )

    del exp_model, exp_tokenizer
    gc.collect()

    return {
        "param_match": param_match,
        "config_pass": config_pass,
        "token_pass": token_pass,
        **numeric_results,
    }


# -------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------


def print_summary(results):
    """Print final summary."""
    config_pass = results.get("config_pass", False)
    token_pass = results.get("token_pass", False)
    text_pass = results.get("text_pass", False)
    param_match = results.get("param_match", False)
    text_gen_match = results.get("text_gen_match", None)

    all_pass = config_pass and token_pass and text_pass
    print("\n" + "=" * 70)
    print(
        "  ✅ ALL CHECKS PASSED"
        if all_pass
        else "  ❌ SOME CHECKS FAILED — Review output above"
    )
    print(f"     - Config fields match {'✓' if config_pass else '✗'}")
    print(f"     - Token IDs match     {'✓' if token_pass else '✗'}")
    print(
        f"     - Parameter count:    {'match ✓' if param_match else 'differ ✗'}"
    )
    print(
        f"     - Text logit parity   {'✓' if text_pass else '✗'} "
        f"(mean diff: {results.get('text_mean_diff', float('nan')):.2e})"
    )
    if text_gen_match is True:
        print("     - Text generation:    IDENTICAL ✓")
    elif text_gen_match is False:
        print("     - Text generation:    differs (bf16→f32 precision)")
    print("=" * 70 + "\n")
    return all_pass


# -------------------------------------------------------------------
# Entry point
# -------------------------------------------------------------------


def run_verification(
    preset="llama3.2_1b",
    hf_model_id=None,
    export_dir=None,
    skip_generation=False,
):
    """Run the full Llama3 export verification pipeline."""
    hf_model_id = hf_model_id or PRESET_TO_HF.get(preset)
    if hf_model_id is None:
        print(f"Error: No HF model ID for preset '{preset}'.")
        print(f"Available presets: {', '.join(PRESET_TO_HF.keys())}")
        return False

    export_dir = export_dir or tempfile.mkdtemp()
    export_path = os.path.join(export_dir, "llama3_exported")
    os.makedirs(export_path, exist_ok=True)

    print("\n" + "=" * 70)
    print("  Llama3 Export Verification (Real Pretrained Weights)")
    print("=" * 70)
    print(f"  KerasHub preset : {preset}")
    print(f"  HF model ID     : {hf_model_id}")
    print(f"  Export path     : {export_path}")
    print(f"  Skip generation : {skip_generation}")

    try:
        export_keras_model(preset, export_path)
        original_results = precompute_original_outputs(
            hf_model_id, skip_generation
        )
        validation_results = validate_exported_model(
            export_path, hf_model_id, original_results, skip_generation
        )
        return print_summary(validation_results)
    except KeyboardInterrupt:
        print("\n\n⚠ Verification interrupted by user")
        return False
    except Exception as e:
        import traceback

        print("\n\n❌ Verification failed with error:")
        print(f"   {type(e).__name__}: {e}")
        traceback.print_exc()
        return False


print("✅ Verification functions loaded!")

## Step 5: Run Quick Verification

Start with the **smallest model** for a fast sanity check.

### Available Presets:
| Preset | HF Model | Size |
|--------|----------|------|
| `llama3.2_1b` | meta-llama/Llama-3.2-1B | **1B** (fastest) |
| `llama3.2_instruct_1b` | meta-llama/Llama-3.2-1B-Instruct | 1B |
| `llama3.2_3b` | meta-llama/Llama-3.2-3B | 3B |
| `llama3.2_instruct_3b` | meta-llama/Llama-3.2-3B-Instruct | 3B |
| `llama3_8b_en` | meta-llama/Meta-Llama-3-8B | 8B |
| `llama3.1_8b` | meta-llama/Llama-3.1-8B | 8B |

In [ ]:
# Quick test with the smallest model
success = run_verification(
    preset="llama3.2_1b",
    skip_generation=True,  # Faster — logit check only
    export_dir="/content/llama3_export",
)

if success:
    print("\n🎉 Verification completed successfully!")
else:
    print("\n⚠️ Verification failed. Check the output above for details.")

## Step 6: Run Full Verification (Optional)

Include generation comparison for a more thorough check.

In [ ]:
# Full verification with generation comparison
success = run_verification(
    preset="llama3.2_1b",
    skip_generation=False,  # Include generation validation
    export_dir="/content/llama3_export_full",
)

if success:
    print("\n🎉 Full verification completed successfully!")
else:
    print("\n⚠️ Verification failed. Check the output above for details.")

## Step 7: Test with a Larger Model (Optional)

Verify an 8B model — requires ~16 GB RAM.

In [ ]:
# Verify Llama 3.1-8B (scaled RoPE model)
success = run_verification(
    preset="llama3.1_8b",
    skip_generation=True,
    export_dir="/content/llama3_8b_export",
)

if success:
    print("\n🎉 8B model verification passed!")
else:
    print("\n⚠️ Verification failed. Check the output above for details.")

## Step 8: Inspect Exported Files (Optional)

View the exported HuggingFace model files.

In [ ]:
import json

export_path = "/content/llama3_export/llama3_exported"

print("Exported files:")
!ls -lh {export_path}

print("\nconfig.json (first 15 fields):")
with open(f"{export_path}/config.json") as f:
    cfg = json.load(f)
for k, v in list(cfg.items())[:15]:
    print(f"  {k}: {v}")

print("\ntokenizer_config.json:")
with open(f"{export_path}/tokenizer_config.json") as f:
    tok_cfg = json.load(f)
for k, v in tok_cfg.items():
    print(f"  {k}: {v}")

## Step 9: Clean Up (Optional)

Remove exported files to free up disk space.

In [ ]:
!rm -rf /content/llama3_export* /content/llama3_8b_export*
print("✅ Cleaned up exported files")

## Troubleshooting

### Issue: 401 / 403 Authentication Error
- Make sure you've logged in with a valid HF token (Step 2)
- Ensure you've accepted the Llama3 license at [huggingface.co/meta-llama](https://huggingface.co/meta-llama)
- Each model variant (3.0 / 3.1 / 3.2) may have a separate license to accept

### Issue: Out of Memory
- Use `llama3.2_1b` (smallest model) with `skip_generation=True`
- Restart the runtime and try again
- Make sure you're using a GPU runtime (Runtime → Change runtime type → T4 GPU)

### Issue: Logit parity fails (mean diff ≥ 0.1)
- This may indicate a weight-mapping bug; open an issue with the diff value
- Verify you are comparing the same model variant (e.g. don't mix 3.0 and 3.1)

### Issue: Generation differs
- Minor differences are **expected** when the original model was stored in bf16 but we load in f32
- Identical generation is not required for the overall check to pass (logit parity is the signal)

## Available Presets Reference

In [ ]:
print("Available Llama3 Presets:\n")
for preset, hf_id in PRESET_TO_HF.items():
    print(f"  {preset:30s} → {hf_id}")